# Lab 5: Linear Regression — Predicting House Prices
**Machine Learning — Lab Manual | BS Computer Science, 5th Semester**

| Field | Details |
|---|---|
| **Name** | Abdul Wali |
| **Student ID** | *023-24-0174* |
| **Section** | *F* |
| **GitHub** | *AWS9760* |
| **Dataset** | House Prices — Advanced Regression Techniques (Ames, Iowa) |
| **Dataset Source** | https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques |

## Setup: libraries and helpers
Imports, display options and the random seed used everywhere so results are reproducible.
(Prices in the written text are shown as `USD 12,345` instead of with a dollar sign, because Colab reads `$...$` as maths.)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_columns = 60
pd.options.display.max_colwidth = None
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})


def usd(value, decimals=0):
    """Format a number as 'USD 12,345' (used in the written interpretations)."""
    sign = '-' if value < 0 else ''
    return f"{sign}USD {abs(value):,.{decimals}f}"


def thousands(x, _pos=None):
    """Axis tick formatter: 250000 -> '250k'."""
    return f"{x / 1000:,.0f}k"

## 1. Problem Definition

**Question.** Can we estimate what a residential house in Ames, Iowa will sell for, using a handful of facts that are easy to know about the property (size, quality, age, garage, neighborhood, ...)?

**Task.** Build a *Multiple Linear Regression* model that predicts the target `SalePrice` (US dollars) from 12 selected features. The model learns from `train.csv` (1,460 sold houses) and is finally used to predict the prices of the unlabeled houses in `test.csv` for a Kaggle submission.

**Why regression and not classification?** `SalePrice` is a continuous number (it can be any dollar amount), not one of a fixed set of classes, so the model must output a quantity instead of a label. That changes the evaluation: accuracy and confusion matrices no longer apply, and we instead measure *how far* predictions are from the true prices (MAE, MSE, RMSE), *how much* of the price variation is explained (R²), and we inspect the residuals.

## 2. Dataset Exploration

**Getting the data.** Join the competition on Kaggle, download `train.csv`, `test.csv` and `sample_submission.csv` from the *Data* tab, and upload them when the loader cell below asks (or put them in the Colab session / mounted Drive beforehand). The next cell is an *optional* alternative that downloads them with the Kaggle API.

In [ ]:
# OPTIONAL: download the data with the Kaggle API instead of uploading it by hand.
# Needs a kaggle.json token (Kaggle -> Settings -> API -> Create New Token)
# and the competition rules accepted. Leave False if you are uploading the CSVs manually.
USE_KAGGLE_API = False

if USE_KAGGLE_API:
    from google.colab import files
    files.upload()                                   # choose kaggle.json
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    os.system('cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')
    os.system('kaggle competitions download -c house-prices-advanced-regression-techniques')
    os.system('unzip -o house-prices-advanced-regression-techniques.zip')

In [ ]:
DATA_FILES = ['train.csv', 'test.csv', 'sample_submission.csv']
SEARCH_DIRS = ['.', '/content', '/content/drive/MyDrive',
               '/kaggle/input/house-prices-advanced-regression-techniques']


def locate_data():
    for folder in SEARCH_DIRS:
        if all(os.path.exists(os.path.join(folder, f)) for f in DATA_FILES):
            return folder
    return None


DATA_DIR = locate_data()
if DATA_DIR is None:
    try:
        from google.colab import files
        print("Select train.csv, test.csv and sample_submission.csv from your computer ...")
        files.upload()
    except ImportError:
        pass
    DATA_DIR = locate_data()

if DATA_DIR is None:
    raise FileNotFoundError("train.csv, test.csv and sample_submission.csv were not found. "
                            "Upload them to the Colab session (or set USE_KAGGLE_API = True) and re-run.")
print("Using data from:", os.path.abspath(DATA_DIR))

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print("train.csv             :", train.shape)
print("test.csv              :", test.shape)
print("sample_submission.csv :", sample_sub.shape)

In [ ]:
# First rows
display(train.head())

In [ ]:
# Column names, dtypes and non-null counts
train.info()

In [ ]:
# Summary statistics of all numeric columns
display(train.describe().T)

In [ ]:
n_rows, n_cols = train.shape
n_num = train.select_dtypes(include='number').shape[1]
n_txt = n_cols - n_num
missing = train.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
top_missing = ", ".join(f"`{c}` ({n:,})" for c, n in missing.head(5).items())

display(Markdown(f"""**Interpretation.**

- `train.csv` holds **{n_rows:,} labeled houses** and **{n_cols} columns** (an `Id`, the target `SalePrice` and 79 explanatory features): {n_num} numeric and {n_txt} text (categorical) columns. `test.csv` has {test.shape[0]:,} houses and no `SalePrice`.
- {len(missing)} columns contain missing values; the worst are {top_missing}. In this dataset many of these `NaN`s mean "the house does not have this feature" (no alley, no pool, no fence, ...) rather than "unknown", according to the data description.
- Because the numeric columns live on very different scales (years around 1900-2010, areas in the thousands of square feet, ratings from 1 to 10), their raw coefficients will not be directly comparable later. We only need to worry about the missing values of the features we actually select (checked in Section 3).
"""))

In [ ]:
price = train['SalePrice']
skew_raw = price.skew()
skew_log = np.log(price).skew()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(price, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(price.mean(), color='red', ls='--', label=f'Mean {usd(price.mean())}')
axes[0].axvline(price.median(), color='green', ls='--', label=f'Median {usd(price.median())}')
axes[0].set_title('Distribution of SalePrice')
axes[0].set_xlabel('SalePrice (USD)')
axes[0].set_ylabel('Number of houses')
axes[0].xaxis.set_major_formatter(FuncFormatter(thousands))
axes[0].legend()

axes[1].hist(np.log(price), bins=40, color='darkorange', edgecolor='white')
axes[1].set_title('Distribution of log(SalePrice)')
axes[1].set_xlabel('log(SalePrice)')
axes[1].set_ylabel('Number of houses')
plt.tight_layout()
plt.show()

In [ ]:
side = 'above' if price.mean() > price.median() else 'below'
if abs(skew_log) < abs(skew_raw):
    log_effect = f"makes the distribution much more symmetric (skewness {skew_raw:.2f} -> {skew_log:.2f})"
else:
    log_effect = f"does not make the distribution more symmetric (skewness {skew_raw:.2f} -> {skew_log:.2f})"

display(Markdown(f"""**Interpretation of the plots.** The middle half of the houses sells for between {usd(price.quantile(0.25))} and {usd(price.quantile(0.75))}, while a thin tail of expensive houses stretches the range up to {usd(price.max())}. The mean ({usd(price.mean())}) therefore sits {side} the median ({usd(price.median())}); the skewness of {skew_raw:.2f} confirms the asymmetry. Taking the logarithm (right plot) {log_effect}. This matters later: a straight-line model treats a USD error on a cheap house the same as on an expensive one, and Kaggle scores submissions on log prices so that *relative* (percentage) errors count more than absolute dollar errors.
"""))

## 3. Feature and Target Identification

- **Dependent variable (target):** `SalePrice`, the final sale price in US dollars.
- **Independent variables:** out of the 79 raw features I selected **12** that I would reasonably expect to drive price — **8 numerical** and **4 categorical** (full feature engineering of all 79 is out of scope for this lab).

In [ ]:
TARGET = 'SalePrice'

NUMERIC_FEATURES = ['GrLivArea', 'OverallQual', 'TotalBsmtSF', 'GarageCars',
                    'YearBuilt', 'FullBath', 'LotArea', 'Fireplaces']
CATEGORICAL_FEATURES = ['Neighborhood', 'HouseStyle', 'KitchenQual', 'CentralAir']

print("Dependent variable  :", TARGET)
print("Numerical features  :", NUMERIC_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)

feature_info = pd.DataFrame([
    ('GrLivArea',    'Numerical',                      'Above-ground living area (sq ft)',       'More living space should mean a higher price'),
    ('OverallQual',  'Numerical (rating 1-10)',        'Overall material and finish quality',    'Better build quality commands a premium'),
    ('TotalBsmtSF',  'Numerical',                      'Total basement area (sq ft)',            'Extra usable space'),
    ('GarageCars',   'Numerical',                      'Garage capacity (number of cars)',       'A garage is a valued amenity'),
    ('YearBuilt',    'Numerical',                      'Original construction year',             'Newer houses are usually worth more'),
    ('FullBath',     'Numerical',                      'Number of full bathrooms',               'Bathrooms are a key buyer criterion'),
    ('LotArea',      'Numerical',                      'Lot size (sq ft)',                       'Land itself has value'),
    ('Fireplaces',   'Numerical',                      'Number of fireplaces',                   'Comfort / luxury feature'),
    ('Neighborhood', f"Categorical (nominal, {train['Neighborhood'].nunique()} areas)",
                                                       'Physical location within Ames',          'Location is a classic price driver'),
    ('HouseStyle',   f"Categorical (nominal, {train['HouseStyle'].nunique()} styles)",
                                                       'Style of dwelling (1Story, 2Story, ...)', 'Style relates to layout and size'),
    ('KitchenQual',  'Categorical (ordinal: Po < Fa < TA < Gd < Ex)',
                                                       'Kitchen quality',                        'Kitchens strongly shape buyer perception'),
    ('CentralAir',   'Categorical (binary: Y / N)',    'Central air conditioning',               'A common expectation that affects value'),
], columns=['Feature', 'Type', 'Description', 'Why I expect it to matter'])
display(feature_info)

In [ ]:
# Summary statistics of the chosen numerical features and the target
display(train[NUMERIC_FEATURES + [TARGET]].describe().T)

# Overview of the categorical features
cat_summary = pd.DataFrame({
    'Distinct categories': [train[c].nunique() for c in CATEGORICAL_FEATURES],
    'Most frequent category': [train[c].mode()[0] for c in CATEGORICAL_FEATURES],
    'Share of houses in it (%)': [100 * (train[c] == train[c].mode()[0]).mean() for c in CATEGORICAL_FEATURES],
}, index=CATEGORICAL_FEATURES)
display(cat_summary)

# Missing values in the chosen columns (train vs test)
chosen = NUMERIC_FEATURES + CATEGORICAL_FEATURES
missing_chosen = pd.DataFrame({
    'Missing in train.csv': train[chosen].isna().sum(),
    'Missing in test.csv': test[chosen].isna().sum(),
})
display(missing_chosen.T)

In [ ]:
gaps_test = missing_chosen[missing_chosen['Missing in test.csv'] > 0]
gaps_train = missing_chosen[missing_chosen['Missing in train.csv'] > 0]
if len(gaps_train) == 0 and len(gaps_test) == 0:
    gap_txt = "None of the 12 chosen features has missing values in either file."
elif len(gaps_train) == 0:
    names = ", ".join(f"`{n}` ({int(v)})" for n, v in gaps_test['Missing in test.csv'].items())
    gap_txt = f"`train.csv` has no missing values in the 12 chosen columns, but `test.csv` has a few gaps: {names}. They will be filled with training-set medians / modes in Section 4 so that every test house still gets a prediction."
else:
    names = ", ".join(f"`{n}`" for n in gaps_train.index)
    gap_txt = f"Missing values exist in the chosen columns ({names}); they will be filled in Section 4."

num_std = train[NUMERIC_FEATURES].std()
biggest_scale = num_std.idxmax()
smallest_scale = num_std.idxmin()

display(Markdown(f"""**Interpretation.**

- The 12 features cover size (`GrLivArea`, `TotalBsmtSF`, `LotArea`), quality (`OverallQual`, `KitchenQual`), age (`YearBuilt`), amenities (`GarageCars`, `FullBath`, `Fireplaces`, `CentralAir`) and location / type (`Neighborhood`, `HouseStyle`).
- {gap_txt}
- The numeric features are on very different scales: the standard deviation of `{biggest_scale}` is about {num_std[biggest_scale]:,.1f}, while for `{smallest_scale}` it is only {num_std[smallest_scale]:,.2f}. This is why raw coefficients can't be compared directly (Section 12).
- Each categorical feature has a different number of categories (see the table), and none of them can be fed to `LinearRegression` as text — they need encoding (Section 4).
"""))

In [ ]:
# Correlation of each numerical feature with the target
corr = train[NUMERIC_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(corr.index, corr.values, color='steelblue')
for i, v in enumerate(corr.values):
    ax.text(v + 0.01 if v >= 0 else v - 0.01, i, f'{v:.2f}', va='center', ha='left' if v >= 0 else 'right')
ax.set_xlim(min(0, corr.min()) - 0.1, 1)
ax.set_xlabel('Pearson correlation with SalePrice')
ax.set_title('Numerical features vs. SalePrice')
plt.tight_layout()
plt.show()

# strongest correlation between two *features* (they will share credit in the coefficients)
feat_corr = train[NUMERIC_FEATURES].corr()
best = (None, None, 0.0)
for i, a in enumerate(NUMERIC_FEATURES):
    for b in NUMERIC_FEATURES[i + 1:]:
        if abs(feat_corr.loc[a, b]) > abs(best[2]):
            best = (a, b, feat_corr.loc[a, b])

if abs(best[2]) >= 0.5:
    overlap_txt = f"The features are also related to *each other* — the strongest pair is `{best[0]}` and `{best[1]}` (r = {best[2]:.2f}) — so when the model splits the credit between them, individual coefficients should be read with care."
else:
    overlap_txt = f"The numerical features are only weakly related to each other (the strongest pair, `{best[0]}` and `{best[1]}`, has r = {best[2]:.2f}), so each coefficient should be fairly stable."
pos_txt = ("All of the selected numerical features are positively related to price, which matches the reasoning in the table above."
           if (corr > 0).all() else
           f"{int((corr > 0).sum())} of the {len(corr)} selected numerical features are positively related to price; `{corr.index[0]}` is not, which differs from my initial expectation in the table above.")

display(Markdown(f"""**Interpretation.** `{corr.index[-1]}` has the strongest linear relationship with price (r = {corr.iloc[-1]:.2f}), followed by `{corr.index[-2]}` (r = {corr.iloc[-2]:.2f}); `{corr.index[0]}` is the weakest (r = {corr.iloc[0]:.2f}). {pos_txt} {overlap_txt}
"""))

## 4. Data Preparation

`LinearRegression` needs a fully numeric matrix with no gaps. The encoding is chosen according to the *type* of each categorical column:

| Column | Type | Encoding | Justification |
|---|---|---|---|
| `CentralAir` | binary (Y / N) | map `N -> 0`, `Y -> 1` | Only two values, so one 0/1 column keeps all the information. |
| `KitchenQual` | ordinal (Po < Fa < TA < Gd < Ex) | ordinal map `Po=1 ... Ex=5` | The categories have a natural order; one column preserves it (assumes roughly equal steps between levels). |
| `Neighborhood`, `HouseStyle` | nominal (no order) | one-hot with `drop_first=True` | Integer codes would invent a false order/distance between categories. One-hot gives each category its own 0/1 column; dropping one avoids the *dummy-variable trap* (perfect multicollinearity with the intercept), and the dropped category becomes the reference level. |

`OverallQual` is already a 1-10 rating and is used as a number. Missing values (only a few in `test.csv`) are filled with the **training-set** median (numeric) or mode (categorical), so no information from the test file leaks into the model.

In [ ]:
KITCHEN_ORDER = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}   # ordinal encoding
CENTRAL_AIR_MAP = {'N': 0, 'Y': 1}                              # binary encoding
NOMINAL_FEATURES = ['Neighborhood', 'HouseStyle']              # one-hot encoding

# Fill values are learned from train.csv only
fill_values = {c: train[c].median() for c in NUMERIC_FEATURES}
fill_values.update({c: train[c].mode()[0] for c in CATEGORICAL_FEATURES})


def prepare_features(df, reference_columns=None):
    """Select the 12 features and encode them numerically.

    For train.csv: reference_columns=None (creates the columns).
    For test.csv : pass X.columns so the result has exactly the same columns as the training matrix.
    """
    out = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
    out = out.fillna(fill_values)
    out['CentralAir'] = out['CentralAir'].map(CENTRAL_AIR_MAP)
    out['KitchenQual'] = out['KitchenQual'].map(KITCHEN_ORDER)
    if reference_columns is None:
        out = pd.get_dummies(out, columns=NOMINAL_FEATURES, drop_first=True, dtype=int)
    else:
        out = pd.get_dummies(out, columns=NOMINAL_FEATURES, drop_first=False, dtype=int)
        out = out.reindex(columns=reference_columns, fill_value=0)
    return out


X = prepare_features(train)
y = train[TARGET]

# reference (dropped) category of each one-hot encoded feature = first one alphabetically
REFERENCE = {c: sorted(train[c].unique())[0] for c in NOMINAL_FEATURES}
for c, ref in REFERENCE.items():
    assert f"{c}_{ref}" not in X.columns, f"unexpected reference category for {c}"

print("Reference categories (dropped):", REFERENCE)
display(X.head())

In [ ]:
all_numeric = all(pd.api.types.is_numeric_dtype(t) for t in X.dtypes)
n_missing_X = int(X.isna().sum().sum())
n_missing_y = int(y.isna().sum())

print("X shape                 :", X.shape)
print("All columns numeric     :", all_numeric)
print("Missing values in X     :", n_missing_X)
print("Missing values in y     :", n_missing_y)
assert all_numeric and n_missing_X == 0 and n_missing_y == 0

In [ ]:
n_dummies = {c: sum(col.startswith(c + '_') for col in X.columns) for c in NOMINAL_FEATURES}
display(Markdown(f"""**Interpretation.** The prepared matrix `X` has **{X.shape[0]:,} rows and {X.shape[1]} columns**: the 8 numerical features, the 2 encoded single-column features (`CentralAir`, `KitchenQual`) and {sum(n_dummies.values())} dummy columns ({n_dummies['Neighborhood']} for `Neighborhood`, {n_dummies['HouseStyle']} for `HouseStyle`, one fewer than the number of categories because the reference level is dropped). All columns are numeric and there are **no missing values** in `X` or `y`, so the data is ready for `LinearRegression`. The 12 original features have expanded into {X.shape[1]} model inputs, which is normal for one-hot encoding.
"""))

## 5. Train/Test Split
A test set is held out **before** any model is trained. It plays the role of "houses the model has never seen", so the scores on it are an honest estimate of real-world performance. I use an 80/20 split with a fixed `random_state` for reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

split_table = pd.DataFrame({
    'Set': ['X_train', 'X_test', 'y_train', 'y_test'],
    'Shape': [str(X_train.shape), str(X_test.shape), str(y_train.shape), str(y_test.shape)],
    'Share of data (%)': [f"{100 * len(s) / len(X):.1f}" for s in (X_train, X_test, y_train, y_test)],
    'Mean SalePrice (USD)': ['-', '-', f"{y_train.mean():,.0f}", f"{y_test.mean():,.0f}"],
})
display(split_table.set_index('Set'))

In [ ]:
display(Markdown(f"""**Interpretation.** The data was split into **{len(X_train):,} training houses** ({100 * len(X_train) / len(X):.0f}%) and **{len(X_test):,} test houses** ({100 * len(X_test) / len(X):.0f}%), both with the same {X_train.shape[1]} feature columns. The average price is similar in the two parts ({usd(y_train.mean())} in training vs. {usd(y_test.mean())} in testing), so the random split did not put systematically cheaper or more expensive houses on one side. From here on, the test set is used **only for evaluation**.
"""))

## 6. Linear Regression Model
Multiple Linear Regression fits a plane

`price = b0 + b1·x1 + b2·x2 + ... + bn·xn`

by choosing the intercept `b0` and coefficients `b1 ... bn` that minimise the sum of squared residuals on the training data (ordinary least squares). It has no hyper-parameters to tune.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Trained on {X_train.shape[0]:,} houses with {X_train.shape[1]} input columns.")
print(f"Intercept (b0)         : {model.intercept_:,.2f}")
print(f"Number of coefficients : {len(model.coef_)}")

In [ ]:
display(Markdown(f"""**Interpretation.** The model was fitted on the {X_train.shape[0]:,} training houses and learned one intercept plus **{len(model.coef_)} coefficients** — one per input column. Nothing about the test houses was used for fitting. What these numbers mean is discussed in Section 8; how well they predict is evaluated in Sections 7 and 10.
"""))

## 7. Predictions
Predicted prices for the held-out test houses, compared with the true prices for 10 of them.

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    'Id': train.loc[y_test.index, 'Id'].values,
    'Actual': y_test.values,
    'Predicted': y_pred,
})
comparison['Error'] = comparison['Actual'] - comparison['Predicted']
comparison['Abs % error'] = 100 * comparison['Error'].abs() / comparison['Actual']

sample10 = comparison.head(10).reset_index(drop=True)
shown = sample10.copy()
for col in ['Actual', 'Predicted', 'Error']:
    shown[col] = shown[col].map('{:,.0f}'.format)
shown['Abs % error'] = shown['Abs % error'].map('{:.1f}%'.format)
shown = shown.rename(columns={'Actual': 'Actual price (USD)',
                              'Predicted': 'Predicted price (USD)',
                              'Error': 'Error = actual - predicted (USD)'})
display(shown)

In [ ]:
within10 = int((sample10['Abs % error'] <= 10).sum())
mean_ape = sample10['Abs % error'].mean()
worst = sample10.loc[sample10['Abs % error'].idxmax()]
worst_dir = 'over-predicted' if worst['Error'] < 0 else 'under-predicted'
n_nonpos = int((y_pred <= 0).sum())
neg_txt = (f" Note that {n_nonpos} of the {len(y_pred)} test predictions are zero or negative, which is impossible for a price — a known weakness of linear models at the low end."
           if n_nonpos > 0 else "")

display(Markdown(f"""**Interpretation.** Among the 10 houses shown, **{within10} are predicted within 10% of the true price**, and the average absolute percentage error of these 10 is {mean_ape:.1f}%. The largest miss is house Id {int(worst['Id'])}: actual {usd(worst['Actual'])} vs. predicted {usd(worst['Predicted'])}, i.e. {worst_dir} by {worst['Abs % error']:.1f}%. So the predictions follow the real prices, but individual houses can be off noticeably — unsurprising, since the model sees only 12 facts about each house.{neg_txt} A fair judgement needs metrics over the whole test set (Section 10).
"""))

## 8. Coefficients and Intercept
Unlike a Decision Tree or Random Forest, a linear model shows its "reasoning" directly: each coefficient is the change in predicted price for a one-unit increase in that feature **holding all other features constant**. For one-hot columns it is the price difference relative to the reference category that was dropped.

In [ ]:
NUMERIC_UNITS = {
    'GrLivArea':   'additional square foot of above-ground living area',
    'OverallQual': 'one-point rise in overall quality rating (1-10 scale)',
    'TotalBsmtSF': 'additional square foot of basement area',
    'GarageCars':  'additional car of garage capacity',
    'YearBuilt':   'year newer the house was built',
    'FullBath':    'additional full bathroom',
    'LotArea':     'additional square foot of lot area',
    'Fireplaces':  'additional fireplace',
}


def money_amount(v):
    a = abs(v)
    return usd(a, 2) if a < 100 else usd(a, 0)


def interpret(feature, coef):
    amount = money_amount(coef)
    up_down = 'increase' if coef >= 0 else 'decrease'
    hi_lo = 'higher' if coef >= 0 else 'lower'
    if feature in NUMERIC_UNITS:
        return (f"Each {NUMERIC_UNITS[feature]} is associated with a price {up_down} of about {amount}, "
                f"holding the other features constant.")
    if feature == 'CentralAir':
        return f"A house with central air is priced about {amount} {hi_lo} than an otherwise similar house without it."
    if feature == 'KitchenQual':
        return (f"Each one-step better kitchen quality (Po < Fa < TA < Gd < Ex) is associated with a price {up_down} "
                f"of about {amount}, holding the other features constant.")
    group, category = feature.split('_', 1)
    return (f"A house with {group} = '{category}' is priced about {amount} {hi_lo} than an otherwise similar house "
            f"in the reference {group} '{REFERENCE[group]}'.")


coef_table = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': model.coef_})
coef_table['Interpretation'] = [interpret(f, c) for f, c in zip(coef_table['Feature'], coef_table['Coefficient'])]

print(f"Intercept (b0): {model.intercept_:,.2f} USD")
display(coef_table)

In [ ]:
b0 = model.intercept_
yr_coef = coef_table.set_index('Feature').loc['YearBuilt', 'Coefficient']
yr_mean = X_train['YearBuilt'].mean()
yr_contrib = yr_coef * yr_mean
sign_txt = ("Here it is even **negative**, which would be an impossible negative price." if b0 < 0
            else "Here it is positive, but that number still does not describe any real house.")

coef_lookup = coef_table.set_index('Feature')['Coefficient']
expected_positive = NUMERIC_FEATURES + ['KitchenQual', 'CentralAir']
neg_expected = [f for f in expected_positive if coef_lookup[f] < 0]
if not neg_expected:
    sign_check_txt = "All numerical features and the two ordered / binary features have positive coefficients, exactly as common sense predicts (bigger, better-quality, newer houses cost more)."
else:
    names = ", ".join(f"`{f}` ({usd(coef_lookup[f])})" for f in neg_expected)
    sign_check_txt = (f"Most of these features have the positive sign common sense predicts, but {names} came out **negative**. That does not mean the feature lowers value: it is most likely caused by correlation with other features "
                      f"(e.g. once living area and quality are in the model, extra bathrooms or fireplaces add little or slightly negative value on their own).")

display(Markdown(f"""**Interpretation of the coefficients.** The table reads directly as "extra price per unit, other things equal": for example, {coef_table.loc[0, 'Interpretation'][0].lower() + coef_table.loc[0, 'Interpretation'][1:]} {sign_check_txt} Overall {int((coef_table['Coefficient'] < 0).sum())} of the {len(coef_table)} coefficients are negative, mostly neighborhood / house-style dummies that are cheaper than their reference category. A negative coefficient does not always mean "this feature lowers value": for dummy columns it only says that category is cheaper than the reference category, and for correlated features (e.g. size variables) the model may shift credit from one to another. Coefficients describe *association in this data*, not cause and effect.

**Does the intercept have a real-world meaning?** The intercept is **{usd(b0)}**. It is the predicted price of a house for which *every* feature equals 0: built in the year 0, zero living area, no basement, no lot, a quality rating and kitchen quality of 0 (both scales start at 1), no central air, and belonging to the reference neighborhood and house style. No such house exists and it lies far outside the data (`YearBuilt` ranges only from {int(train['YearBuilt'].min())} to {int(train['YearBuilt'].max())}), so the intercept has **no sensible real-world interpretation**. {sign_txt} It is simply the constant that positions the fitted plane at the right height — for instance, at the average build year ({yr_mean:.0f}) the `YearBuilt` term alone contributes {usd(yr_contrib)}, which the intercept has to balance out.
"""))

## 9. Residual Analysis
Residual = actual price − predicted price. A **positive** residual means the model *under*-predicted; a **negative** one means it *over*-predicted. If a linear model is appropriate, residuals should look like random noise: centred on 0, with roughly constant spread, and no visible curve.

In [ ]:
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].scatter(y_pred, residuals, alpha=0.6, edgecolor='k', linewidth=0.3)
axes[0].axhline(0, color='red', ls='--')
axes[0].set_title('Residuals vs. predicted price')
axes[0].set_xlabel('Predicted price (USD)')
axes[0].set_ylabel('Residual = actual - predicted (USD)')
axes[0].xaxis.set_major_formatter(FuncFormatter(thousands))
axes[0].yaxis.set_major_formatter(FuncFormatter(thousands))

axes[1].hist(residuals, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', ls='--')
axes[1].set_title('Distribution of residuals')
axes[1].set_xlabel('Residual (USD)')
axes[1].set_ylabel('Number of houses')
axes[1].xaxis.set_major_formatter(FuncFormatter(thousands))

axes[2].scatter(y_test, y_pred, alpha=0.6, edgecolor='k', linewidth=0.3)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[2].plot(lims, lims, color='red', ls='--', label='Perfect prediction')
axes[2].set_title('Actual vs. predicted price')
axes[2].set_xlabel('Actual price (USD)')
axes[2].set_ylabel('Predicted price (USD)')
axes[2].xaxis.set_major_formatter(FuncFormatter(thousands))
axes[2].yaxis.set_major_formatter(FuncFormatter(thousands))
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
res = pd.Series(residuals)
pred = pd.Series(y_pred)
thirds = pd.qcut(pred, 3, labels=['lowest third', 'middle third', 'highest third'])
by_third = res.groupby(thirds, observed=True).agg(['mean', 'std'])
by_third.columns = ['Mean residual (USD)', 'Std of residuals (USD)']
by_third.index.name = 'Predicted price range'
display(by_third)

In [ ]:
res_mean, res_std, res_skew = res.mean(), res.std(), res.skew()
m_low, m_mid, m_high = by_third['Mean residual (USD)']
spread_ratio = by_third.loc['highest third', 'Std of residuals (USD)'] / by_third.loc['lowest third', 'Std of residuals (USD)']
curved = (np.sign(m_low) == np.sign(m_high)) and (np.sign(m_mid) != np.sign(m_low)) and abs(m_mid) > 0.15 * res_std
n_big = int((res.abs() > 3 * res_std).sum())

centre_txt = (f"The mean residual is {usd(res_mean)}, close to zero compared with their spread (std = {usd(res_std)}), so the model is not systematically too high or too low overall."
              if abs(res_mean) < 0.15 * res_std else
              f"The mean residual is {usd(res_mean)} (std = {usd(res_std)}), so on this particular test set the model is on average {'too low' if res_mean > 0 else 'too high'}.")

if spread_ratio > 1.5:
    spread_txt = f"The vertical spread of the residuals **widens as the predicted price grows** (std {usd(by_third.loc['lowest third', 'Std of residuals (USD)'])} in the cheapest third vs. {usd(by_third.loc['highest third', 'Std of residuals (USD)'])} in the most expensive third, a factor of {spread_ratio:.1f}) — a funnel shape called *heteroscedasticity*: expensive houses are predicted much less precisely than cheap ones."
elif spread_ratio < 0.67:
    spread_txt = f"The spread of the residuals **shrinks** as the predicted price grows (factor {spread_ratio:.1f} between the most expensive and cheapest third), so cheaper houses are predicted less precisely."
else:
    spread_txt = f"The spread of the residuals is fairly constant across price levels (factor {spread_ratio:.1f} between the most expensive and cheapest third), so there is no strong funnel shape."

curve_txt = ("The average residual has the same sign at the low and high end but the opposite sign in the middle (see the table), which points to **curvature** — the price does not grow in a perfectly straight line with the features."
             if curved else
             "The average residual per price range (table above) does not show a clear U-shaped or curved pattern.")

if res_skew > 0.5:
    hist_txt = f"The histogram is centred near zero but **right-skewed** (skewness {res_skew:.2f}): a few houses have large positive residuals, i.e. they sold for much more than the model predicted."
elif res_skew < -0.5:
    hist_txt = f"The histogram is centred near zero but **left-skewed** (skewness {res_skew:.2f}): a few houses have large negative residuals, i.e. the model predicted much more than they sold for."
else:
    hist_txt = f"The histogram is roughly symmetric around zero (skewness {res_skew:.2f}), which is what we hope for."

random_like = (spread_ratio <= 1.5 and spread_ratio >= 0.67) and (not curved) and abs(res_skew) <= 0.5
verdict = ("The residuals look **reasonably random**, so the linear assumptions hold up acceptably for these features."
           if random_like else
           "So the residuals are **not perfectly random**: there is a visible pattern, which means a plain straight-line model leaves some structure unexplained.")

display(Markdown(f"""**Interpretation.**

- {centre_txt}
- {spread_txt}
- {curve_txt}
- {hist_txt} Number of test houses with a residual larger than 3 standard deviations (outliers): {n_big}.
- In the right-hand plot, points on the red diagonal would be perfect predictions; the closer the cloud hugs it, the better the model.

{verdict}
"""))

## 10. Regression Metrics
Four standard metrics on the **test set**, next to a naive baseline that predicts the training-set mean price for every house (so we can see whether the model is better than "no model").

In [ ]:
def regression_metrics(y_true, y_hat):
    mse = mean_squared_error(y_true, y_hat)
    return {'MAE': mean_absolute_error(y_true, y_hat),
            'MSE': mse,
            'RMSE': float(np.sqrt(mse)),
            'R²': r2_score(y_true, y_hat)}


def fmt_metric(name, value):
    return f"{value:.4f}" if name == 'R²' else f"{value:,.0f}"


test_metrics = regression_metrics(y_test, y_pred)
baseline_metrics = regression_metrics(y_test, np.full(len(y_test), y_train.mean()))

metrics_table = pd.DataFrame({
    'Metric': list(test_metrics),
    'Linear Regression (test set)': [fmt_metric(k, v) for k, v in test_metrics.items()],
    'Baseline: always predict mean price': [fmt_metric(k, v) for k, v in baseline_metrics.items()],
    'Units': ['USD', 'USD squared', 'USD', 'unitless (1 = perfect)'],
}).set_index('Metric')
display(metrics_table)

In [ ]:
mae, mse, rmse, r2 = (test_metrics[k] for k in ['MAE', 'MSE', 'RMSE', 'R²'])
mae_pct = 100 * mae / y_test.mean()

display(Markdown(f"""**What each metric tells us (in my own words).**

- **MAE = {usd(mae)}.** On average, a prediction misses the true price by about {usd(mae)}, which is roughly {mae_pct:.1f}% of the average test-house price. Every error counts in proportion to its size, so it is the easiest metric to read.
- **MSE = {mse:,.0f} (USD squared).** The average of the *squared* errors. Squaring punishes large misses much more heavily than small ones, but the unit (dollars squared) is not intuitive. It is the quantity that least-squares training minimises, and the basis for RMSE.
- **RMSE = {usd(rmse)}.** The square root of MSE, which brings the error back into dollars. It is {rmse / mae:.2f} times the MAE; the further RMSE is above MAE, the more the error is dominated by a few large misses.
- **R² = {r2:.3f}.** The model explains about **{100 * r2:.1f}%** of the variation in house prices on unseen data, compared with always guessing the mean price (which scores an R² of about 0). The remaining {100 * (1 - r2):.1f}% is variation the 12 features cannot capture.

**Comparison with the baseline.** Predicting the mean price for every house would give an MAE of {usd(baseline_metrics['MAE'])} and an RMSE of {usd(baseline_metrics['RMSE'])}; the regression model cuts these to {usd(mae)} and {usd(rmse)}, so the features clearly add predictive value.

**Which metric for a non-technical stakeholder?** I would show the **MAE**: "on average our price estimate is off by about {usd(mae)}" needs no statistics background, is expressed in the same unit the stakeholder cares about (dollars), and isn't inflated by a few extreme houses. R² ("the model explains {100 * r2:.0f}% of price differences") is a good one-line supplement; MSE and RMSE are better suited for technical model comparison.
"""))

## 11. Training vs Testing Performance
As in the overfitting experiment of Lab 3: the same four metrics on the data the model was trained on vs. data it has never seen. A model that has memorised its training data looks great on the former and much worse on the latter.

In [ ]:
train_pred = model.predict(X_train)
train_metrics = regression_metrics(y_train, train_pred)

rows = []
for k in ['MAE', 'MSE', 'RMSE', 'R²']:
    diff = test_metrics[k] - train_metrics[k]
    diff_txt = f"{diff:+.4f}" if k == 'R²' else f"{diff:+,.0f}"
    rows.append([k, fmt_metric(k, train_metrics[k]), fmt_metric(k, test_metrics[k]), diff_txt])
train_test_table = pd.DataFrame(rows, columns=['Metric', 'Training set', 'Test set', 'Test - Training']).set_index('Metric')
display(train_test_table)

In [ ]:
r2_tr, r2_te = train_metrics['R²'], test_metrics['R²']
rmse_tr, rmse_te = train_metrics['RMSE'], test_metrics['RMSE']
r2_gap = r2_tr - r2_te
rmse_ratio = rmse_te / rmse_tr

if r2_gap > 0.10 or rmse_ratio > 1.30:
    verdict11 = (f"**Yes — the gap is large.** R² falls from {r2_tr:.3f} to {r2_te:.3f} and the RMSE grows by a factor of {rmse_ratio:.2f} on unseen houses. This is the classic sign of **overfitting**: the model has fitted quirks of the training houses that do not carry over to new ones.")
elif r2_gap < -0.03:
    verdict11 = (f"**No overfitting is visible; the test score is even higher.** R² is {r2_te:.3f} on the test set vs. {r2_tr:.3f} on the training set (RMSE ratio test/train = {rmse_ratio:.2f}). A better test score is not a sign of a better model: it means this particular random split happened to contain slightly easier houses (for example fewer extreme outliers) than the training part.")
else:
    verdict11 = (f"**No — the gap is small.** R² is {r2_tr:.3f} on the training set and {r2_te:.3f} on the test set (difference {r2_gap:+.3f}), and the test RMSE is {rmse_ratio:.2f} times the training RMSE. The model performs about as well on new houses as on the ones it learned from, so its performance is **consistent and it is not overfitting**.")

fit_txt = (f" Both scores are fairly low (R² below 0.7), so the bigger issue is *underfitting*: the 12 features and a straight-line model cannot capture everything that drives price."
           if max(r2_tr, r2_te) < 0.7 else
           f" The remaining error is therefore not caused by overfitting but by what the model cannot express — limited features and a purely linear form.")

display(Markdown(f"""**Interpretation.** A *large* gap (high training R² but much lower test R², and test errors far above training errors) would mean the model memorised the training data and would not generalize. **Is there such a gap here?** {verdict11}{fit_txt} This is expected for a plain linear model with only {X_train.shape[1]} inputs and {X_train.shape[0]:,} training rows: it has few degrees of freedom compared with, say, an unpruned Decision Tree from Lab 3, so it has little room to overfit.
"""))

## 12. Model Interpretation
Which features matter most? Comparing raw coefficients is misleading: a coefficient is "price change **per unit**", and a unit means very different things (one square foot, one quality point, one year, one 0/1 dummy). To make features comparable I also compute the effect of a **one-standard-deviation** change: `coefficient × standard deviation of the feature`.

In [ ]:
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_,
    'Std dev of feature': X_train.std().values,
})
importance['Effect of +1 SD (USD)'] = importance['Coefficient'] * importance['Std dev of feature']
importance['Abs effect'] = importance['Effect of +1 SD (USD)'].abs()

top_raw = importance.loc[importance['Coefficient'].abs().sort_values(ascending=False).index].head(8)
top_std = importance.sort_values('Abs effect', ascending=False).head(8)

print("Top 8 by RAW coefficient size:")
display(top_raw[['Feature', 'Coefficient', 'Std dev of feature']].reset_index(drop=True))
print("Top 8 by effect of a ONE-STANDARD-DEVIATION change:")
display(top_std[['Feature', 'Coefficient', 'Std dev of feature', 'Effect of +1 SD (USD)']].reset_index(drop=True))

In [ ]:
plot_df = importance.sort_values('Abs effect', ascending=False).head(12).iloc[::-1]
colors = ['seagreen' if v >= 0 else 'indianred' for v in plot_df['Effect of +1 SD (USD)']]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot_df['Feature'], plot_df['Effect of +1 SD (USD)'], color=colors)
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('Change in predicted price for a +1 standard-deviation change (USD)')
ax.set_title('Standardized feature effects (top 12)')
ax.xaxis.set_major_formatter(FuncFormatter(thousands))
plt.tight_layout()
plt.show()

In [ ]:
top3 = top_std.head(3)
raw3 = list(top_raw['Feature'].head(3))
std3 = list(top3['Feature'])
same = set(raw3) == set(std3)
dummy_like = [f for f in raw3 if f not in NUMERIC_FEATURES + ['KitchenQual', 'CentralAir']]

top3_parts = []
for i in range(len(top3)):
    feat = top3.iloc[i]['Feature']
    eff = top3.iloc[i]['Effect of +1 SD (USD)']
    top3_parts.append(f"**{i + 1}. `{feat}`** ({'+' if eff >= 0 else '-'}{usd(abs(eff))} per +1 SD)")
top3_txt = "; ".join(top3_parts)

if same:
    raw_txt = f"The three largest *raw* coefficients ({', '.join('`' + f + '`' for f in raw3)}) are the same features, though not necessarily in the same order."
else:
    if len(dummy_like) == len(raw3):
        dummy_note = " All of them are 0/1 dummy columns, so each coefficient is a price gap between two groups of houses, not the effect of a continuous quantity."
    elif dummy_like:
        dummy_note = f" {len(dummy_like)} of them are 0/1 dummy columns, so their coefficients are price gaps between two groups of houses, not the effect of a continuous quantity."
    else:
        dummy_note = ""
    raw_txt = (f"The three largest *raw* coefficients belong to {', '.join('`' + f + '`' for f in raw3)}, which is **not** the same set as the most influential features below — raw size mostly reflects the unit a feature is measured in.{dummy_note}")

display(Markdown(f"""**Interpretation.** {raw_txt} After scaling each coefficient by the spread of its feature, the three most influential features on price are: {top3_txt}.

A larger coefficient does **not** automatically mean a more important feature when the underlying scales differ: `GrLivArea` has a small coefficient per square foot but ranges over thousands of square feet, whereas a dummy like a single neighborhood can have a huge coefficient but applies to only a small share of houses. The standardized effect answers a fairer question — "how much does the predicted price move for a typical change in this feature?".

Two cautions remain: (1) the effects are *associations*, not causes; (2) correlated features (size, quality and garage capacity tend to rise together) share credit, so an individual coefficient depends on which other features are in the model.
"""))

## 13. Conclusion

### 13.1 Kaggle submission file
The **same feature selection and encoding** (`prepare_features`) is applied to `test.csv`. For the final submission the model is re-fitted on **all 1,460 labeled houses** (no reason to waste the 20% that was held out for evaluation). Because a linear model can, in rare cases, predict a price below zero — which breaks Kaggle's log-based metric — predictions are clipped at the lowest price seen in training.

In [ ]:
MIN_PRICE = float(y.min())

final_model = LinearRegression().fit(X, y)

X_kaggle = prepare_features(test, reference_columns=X.columns)
assert list(X_kaggle.columns) == list(X.columns)
assert X_kaggle.isna().sum().sum() == 0

raw_kaggle_pred = final_model.predict(X_kaggle)
n_clipped = int((raw_kaggle_pred < MIN_PRICE).sum())
kaggle_pred = np.clip(raw_kaggle_pred, MIN_PRICE, None)

submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': kaggle_pred})
submission.to_csv('submission.csv', index=False)

# validate against sample_submission.csv
checks = {
    'Columns are exactly Id, SalePrice': list(submission.columns) == ['Id', 'SalePrice'] == list(sample_sub.columns),
    f'Row count matches sample_submission ({len(sample_sub):,})': len(submission) == len(sample_sub),
    'Same Ids as sample_submission': sorted(submission['Id']) == sorted(sample_sub['Id']),
    'No missing values': bool(submission.notna().all().all()),
    'All prices positive': bool((submission['SalePrice'] > 0).all()),
}
for name, ok in checks.items():
    print(('OK   ' if ok else 'FAIL ') + name)
assert all(checks.values()), "submission.csv failed validation"

display(submission.head())

try:
    from google.colab import files
    files.download('submission.csv')
except ImportError:
    print("Not running in Colab - submission.csv was saved in the current folder.")

In [ ]:
def rmsle(y_true, y_hat):
    """Kaggle's metric: RMSE between log(predicted) and log(actual) price (lower is better)."""
    return float(np.sqrt(mean_squared_error(np.log(y_true), np.log(np.clip(y_hat, MIN_PRICE, None)))))


holdout_rmsle = rmsle(y_test, y_pred)
clip_txt = ("No prediction had to be clipped." if n_clipped == 0
            else f"{n_clipped} prediction(s) were below the minimum training price and had to be clipped to it.")

display(Markdown(f"""**Interpretation.** `submission.csv` contains **{len(submission):,} rows** with exactly the columns `Id` and `SalePrice`, and it passed all five format checks against `sample_submission.csv`. The predicted prices range from {usd(submission['SalePrice'].min())} to {usd(submission['SalePrice'].max())} with a median of {usd(submission['SalePrice'].median())}, in line with the training prices (median {usd(y.median())}). {clip_txt} Kaggle scores on log prices (RMSLE); on my own 20% hold-out set the same kind of model scores **RMSLE = {holdout_rmsle:.4f}**, which is the number to compare with the public leaderboard score once the file is submitted (lower is better).
"""))

### 13.2 Public leaderboard result
1. Open the competition page → **Submit Predictions** → upload `submission.csv`.
2. Copy the *Public Score* and your position from the **Leaderboard** tab and fill in the table below.

*(Optional: the next cell can submit from the notebook using the Kaggle API — set the flag to `True` after configuring `kaggle.json` as in Section 2.)*

In [ ]:
SUBMIT_WITH_API = False   # True = submit from the notebook (needs kaggle.json + accepted competition rules)

if SUBMIT_WITH_API:
    os.system('kaggle competitions submit -c house-prices-advanced-regression-techniques '
              '-f submission.csv -m "Lab 5 - multiple linear regression, 12 features"')
    os.system('kaggle competitions submissions -c house-prices-advanced-regression-techniques | head -5')

| Item | My result |
|---|---|
| Public leaderboard score (RMSLE, lower is better) | *(fill in after submitting)* |
| Leaderboard rank | *(fill in)* out of *(fill in)* submissions |
| My own hold-out RMSLE (from 13.1) | see the output above |

**Reflection (1-2 sentences):** *(write how it felt to see your model measured against thousands of real submissions)*

### 13.3 Quick check of the "next step": modelling log(SalePrice)
The price distribution is right-skewed, the residuals show a pattern (Section 9) and Kaggle scores log prices — so the cheapest improvement to try is to fit the same model on `log(SalePrice)` and convert the predictions back with `exp`.

In [ ]:
log_model = LinearRegression().fit(X_train, np.log(y_train))
log_pred = np.exp(log_model.predict(X_test))


def summary_row(name, predictions):
    m = regression_metrics(y_test, predictions)
    return [name, f"{m['MAE']:,.0f}", f"{m['RMSE']:,.0f}", f"{m['R²']:.4f}", f"{rmsle(y_test, predictions):.4f}"]


next_step = pd.DataFrame([summary_row('Linear Regression on SalePrice', y_pred),
                          summary_row('Linear Regression on log(SalePrice)', log_pred)],
                         columns=['Model', 'MAE (USD)', 'RMSE (USD)', 'R²', 'RMSLE (Kaggle metric)']).set_index('Model')
display(next_step)

In [ ]:
rmsle_raw, rmsle_log = rmsle(y_test, y_pred), rmsle(y_test, log_pred)
m_log = regression_metrics(y_test, log_pred)

if rmsle_log < rmsle_raw:
    log_verdict = f"The log model lowers the Kaggle metric (RMSLE) from {rmsle_raw:.4f} to {rmsle_log:.4f} ({100 * (rmsle_raw - rmsle_log) / rmsle_raw:.0f}% better)."
    funnel = " and the funnel-shaped residuals" if spread_ratio > 1.5 else ""
    log_why = f"A log target makes the model work in percentages (each coefficient becomes an approximate percentage change in price), which suits the right-skewed price distribution{funnel}."
else:
    log_verdict = f"The log model does not improve the Kaggle metric (RMSLE) on this split ({rmsle_raw:.4f} for the plain model vs. {rmsle_log:.4f})."
    log_why = "So on this particular split the plain dollar-based model is at least as good; the idea would need to be confirmed with cross-validation rather than a single 80/20 split."

display(Markdown(f"""**Interpretation.** {log_verdict} In dollar terms, the MAE changes from {usd(mae)} to {usd(m_log['MAE'])} and the RMSE from {usd(rmse)} to {usd(m_log['RMSE'])}. {log_why} (Converting back with `exp` gives median-type predictions, so the dollar metrics are not a perfectly fair comparison.) The submission above deliberately keeps the plain model required by the lab; this check only tests one idea for the next iteration.
"""))

### 13.4 Final interpretation

In [ ]:
if r2 >= 0.8:
    how_well = "well"
elif r2 >= 0.6:
    how_well = "moderately well"
else:
    how_well = "poorly"

gen_txt = ("training and test scores are close, so there is no sign of overfitting"
           if (r2_gap <= 0.10 and rmse_ratio <= 1.30) else
           "the test score is clearly worse than the training score, which indicates overfitting")

n_huge = int((train['GrLivArea'] > 4000).sum())
issues = []
if spread_ratio > 1.5:
    issues.append("the errors grow with the price level")
if curved:
    issues.append("the residuals show curvature")
if abs(res_skew) > 0.5:
    issues.append("the residuals are skewed with a few large misses")
if len(issues) > 1:
    issues_joined = ", ".join(issues[:-1]) + " and " + issues[-1]
elif issues:
    issues_joined = issues[0]
else:
    issues_joined = ""
issue_txt = f" — and in this data {issues_joined}" if issues else ""
outlier_txt = (f", is sensitive to outliers (e.g. the {n_huge} very large houses with `GrLivArea` above 4,000 sq ft)" if n_huge > 0
               else ", is sensitive to outliers")

log_sentence = (f"in a quick check it lowered the Kaggle metric from {rmsle_raw:.3f} to {rmsle_log:.3f}"
                if rmsle_log < rmsle_raw else
                f"a quick check did not lower the Kaggle metric on this split, {rmsle_raw:.3f} vs. {rmsle_log:.3f}, so it needs confirming with cross-validation")

display(Markdown(f"""**Final interpretation.**

1. The model predicts house prices **{how_well}**: on unseen houses it explains {100 * r2:.1f}% of the variation in `SalePrice` (R² = {r2:.3f}) with a typical error (MAE) of {usd(mae)}, about {mae_pct:.0f}% of the average price.
2. Performance carries over to new data — {gen_txt} (R² {r2_tr:.3f} on training vs. {r2_te:.3f} on test).
3. MAE and RMSE are the most useful metrics because they state the typical miss in dollars (RMSE additionally penalizes big misses), with R² as a one-number summary of fit; for a non-technical audience I would report the MAE.
4. The strongest effect on price comes from `{top3.iloc[0]['Feature']}`: a one-standard-deviation change moves the predicted price by about {usd(abs(top3.iloc[0]['Effect of +1 SD (USD)']))}, ahead of `{top3.iloc[1]['Feature']}` and `{top3.iloc[2]['Feature']}`.
5. Linear Regression is limited here because it assumes a straight-line, additive relationship with constant error spread{issue_txt}; it also uses only 12 of the 79 features{outlier_txt}, cannot capture interactions such as "size matters more in expensive neighborhoods", and can even predict impossible negative prices.
6. Next I would model `log(SalePrice)` ({log_sentence}), add more of the 79 features plus interaction / polynomial terms, use Ridge or Lasso regularization with cross-validation, and compare against the tree-based models from Labs 3-4.
7. The final `submission.csv` ({len(submission):,} rows, columns `Id` and `SalePrice`) is ready for the leaderboard, where my hold-out RMSLE of {holdout_rmsle:.3f} can be compared with the public score recorded in 13.2.
"""))